In [32]:
#import libraries
import psycopg2
import pandas as pd
import numpy as np
import os

In [51]:
output_dir = os.path.join(os.path.dirname(os.getcwd()), "data")
chargingstationdata = pd.read_csv(os.path.join(output_dir,'ev_charging_patterns.csv'))

In [35]:
#shape
chargingstationdata.shape


(1320, 20)

In [52]:
#information
chargingstationdata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1320 entries, 0 to 1319
Data columns (total 20 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   User ID                                   1320 non-null   object 
 1   Vehicle Model                             1320 non-null   object 
 2   Battery Capacity (kWh)                    1320 non-null   float64
 3   Charging Station ID                       1320 non-null   object 
 4   Charging Station Location                 1320 non-null   object 
 5   Charging Start Time                       1320 non-null   object 
 6   Charging End Time                         1320 non-null   object 
 7   Energy Consumed (kWh)                     1254 non-null   float64
 8   Charging Duration (hours)                 1320 non-null   float64
 9   Charging Rate (kW)                        1254 non-null   float64
 10  Charging Cost (USD)                 

In [37]:
#descriptive statistics
chargingstationdata.describe()

,Battery Capacity (kWh),Energy Consumed (kWh),Charging Duration (hours),Charging Rate (kW),Charging Cost (USD),State of Charge (Start %),State of Charge (End %),Distance Driven (since last charge) (km),Temperature (°C),Vehicle Age (years)
count,1320.000000,1254.000000,1320.000000,1254.000000,1320.000000,1320.000000,1320.000000,1254.000000,1320.000000,1320.000000
mean,74.534692,42.642894,2.269377,25.963003,22.551352,49.130012,75.141590,153.596788,15.263591,3.612843
std,20.626914,22.411705,1.061037,14.011326,10.751494,24.074134,17.080580,86.004987,14.831216,2.309824
min,1.532807,0.045772,0.095314,1.472549,0.234317,2.325959,7.604224,0.862361,-10.724770,0.000000
25%,62.000000,23.881193,1.397623,13.856583,13.368141,27.786903,62.053266,79.445335,2.800664,2.000000
50%,75.000000,42.691405,2.258136,25.603799,22.076360,48.241771,75.682496,152.259867,14.630846,4.000000
75%,85.000000,61.206218,3.112806,37.502998,31.646044,69.277921,88.201370,226.073284,27.981810,6.000000
max,193.003074,152.238758,7.635145,97.342255,69.407743,152.489761,177.708666,398.364775,73.169588,11.688592


In [38]:
#finding missing value
chargingstationdata.isnull().sum()

User ID                                      0
Vehicle Model                                0
Battery Capacity (kWh)                       0
Charging Station ID                          0
Charging Station Location                    0
Charging Start Time                          0
Charging End Time                            0
Energy Consumed (kWh)                       66
Charging Duration (hours)                    0
Charging Rate (kW)                          66
Charging Cost (USD)                          0
Time of Day                                  0
Day of Week                                  0
State of Charge (Start %)                    0
State of Charge (End %)                      0
Distance Driven (since last charge) (km)    66
Temperature (°C)                             0
Vehicle Age (years)                          0
Charger Type                                 0
User Type                                    0
dtype: int64

In [39]:
#percentage of missing value
chargingstationdata.isnull().sum()/chargingstationdata.shape[0]*100

User ID                                     0.0
Vehicle Model                               0.0
Battery Capacity (kWh)                      0.0
Charging Station ID                         0.0
Charging Station Location                   0.0
Charging Start Time                         0.0
Charging End Time                           0.0
Energy Consumed (kWh)                       5.0
Charging Duration (hours)                   0.0
Charging Rate (kW)                          5.0
Charging Cost (USD)                         0.0
Time of Day                                 0.0
Day of Week                                 0.0
State of Charge (Start %)                   0.0
State of Charge (End %)                     0.0
Distance Driven (since last charge) (km)    5.0
Temperature (°C)                            0.0
Vehicle Age (years)                         0.0
Charger Type                                0.0
User Type                                   0.0
dtype: float64

In [40]:
chargingstationdata.dropna(subset=['Energy Consumed (kWh)','Charging Rate (kW)', 'Distance Driven (since last charge) (km)'], inplace=True)

In [41]:
#All columns in the dataframe
chargingstationdata.columns

Index(['User ID', 'Vehicle Model', 'Battery Capacity (kWh)',
       'Charging Station ID', 'Charging Station Location',
       'Charging Start Time', 'Charging End Time', 'Energy Consumed (kWh)',
       'Charging Duration (hours)', 'Charging Rate (kW)',
       'Charging Cost (USD)', 'Time of Day', 'Day of Week',
       'State of Charge (Start %)', 'State of Charge (End %)',
       'Distance Driven (since last charge) (km)', 'Temperature (°C)',
       'Vehicle Age (years)', 'Charger Type', 'User Type'],
      dtype='object')

In [42]:
# Convert time columns to datetime
chargingstationdata['Charging Start Time'] = pd.to_datetime(chargingstationdata['Charging Start Time'])
chargingstationdata['Charging End Time'] = pd.to_datetime(chargingstationdata['Charging End Time'])

# Extract more time features
chargingstationdata['Charging Hour'] = chargingstationdata['Charging Start Time'].dt.hour['Charging Hour'] = chargingstationdata['Charging Start Time'].dt.hour['Charging Hour'] = chargingstationdata['Charging Start Time'].dt.hour['Charging Month'] = chargingstationdata['Charging Start Time'].dt.month
chargingstationdata['Charging Day'] = chargingstationdata['Charging Start Time'].dt.day
chargingstationdata['Is Weekend'] = chargingstationdata['Charging Start Time'].dt.dayofweek >= 5

C:\Users\harig\AppData\Local\Temp\ipykernel_27164\1057210582.py:6: SettingWithCopyWarning:

modifications to a property of a datetimelike object are not supported and are discarded. Change values on the original.

C:\Users\harig\AppData\Local\Temp\ipykernel_27164\1057210582.py:6: SettingWithCopyWarning:

modifications to a property of a datetimelike object are not supported and are discarded. Change values on the original.

C:\Users\harig\AppData\Local\Temp\ipykernel_27164\1057210582.py:6: SettingWithCopyWarning:

modifications to a property of a datetimelike object are not supported and are discarded. Change values on the original.

C:\Users\harig\AppData\Local\Temp\ipykernel_27164\1057210582.py:6: SettingWithCopyWarning:

modifications to a property of a datetimelike object are not supported and are discarded. Change values on the original.



In [43]:
chargingstationdata.head()

,User ID,Vehicle Model,Battery Capacity (kWh),Charging Station ID,Charging Station Location,Charging Start Time,Charging End Time,Energy Consumed (kWh),Charging Duration (hours),Charging Rate (kW),...,State of Charge (Start %),State of Charge (End %),Distance Driven (since last charge) (km),Temperature (°C),Vehicle Age (years),Charger Type,User Type,Charging Hour,Charging Day,Is Weekend
0,User_1,BMW i3,108.463007,Station_391,Houston,2024-01-01 00:00:00,2024-01-01 00:39:00,60.712346,0.591363,36.389181,...,29.371576,86.119962,293.602111,27.947953,2.0,DC Fast Charger,Commuter,1,1,False
1,User_2,Hyundai Kona,100.000000,Station_428,San Francisco,2024-01-01 01:00:00,2024-01-01 03:01:00,12.339275,3.133652,30.677735,...,10.115778,84.664344,112.112804,14.311026,3.0,Level 1,Casual Driver,1,1,False
2,User_3,Chevy Bolt,75.000000,Station_181,San Francisco,2024-01-01 02:00:00,2024-01-01 04:48:00,19.128876,2.452653,27.513593,...,6.854604,69.917615,71.799253,21.002002,2.0,Level 2,Commuter,1,1,False
3,User_4,Hyundai Kona,50.000000,Station_327,Houston,2024-01-01 03:00:00,2024-01-01 06:42:00,79.457824,1.266431,32.882870,...,83.120003,99.624328,199.577785,38.316313,1.0,Level 1,Long-Distance Traveler,1,1,False
4,User_5,Hyundai Kona,50.000000,Station_108,Los Angeles,2024-01-01 04:00:00,2024-01-01 05:46:00,19.629104,2.019765,10.215712,...,54.258950,63.743786,203.661847,-7.834199,1.0,Level 1,Long-Distance Traveler,1,1,False


In [44]:
# Charging rate (kW)
chargingstationdata['Charging Rate'] = chargingstationdata['Energy Consumed (kWh)'] / chargingstationdata['Charging Duration (hours)']

# Battery percentage charged
chargingstationdata['SOC Difference'] = chargingstationdata['State of Charge (End %)'] - chargingstationdata['State of Charge (Start %)']

# Efficiency ratio
chargingstationdata['Efficiency Ratio'] = chargingstationdata['SOC Difference'] / chargingstationdata['Energy Consumed (kWh)']

In [45]:
# One-hot encode categorical variables
dchargingstationdataf = pd.get_dummies(chargingstationdata, columns=['Vehicle Model', 'Charging Station Location', 
                                'Time of Day', 'Day of Week', 'Charger Type', 
                                'User Type'])

In [46]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime
import numpy as np

# Load the data (use your actual data file or the sample data)


# Convert time columns to datetime
chargingstationdata['Charging Start Time'] = pd.to_datetime(chargingstationdata['Charging Start Time'])
chargingstationdata['Charging End Time'] = pd.to_datetime(chargingstationdata['Charging End Time'])

# Extract hour from charging time
chargingstationdata['Charging Hour'] = chargingstationdata['Charging Start Time'].dt.hour

# Create time categories
def categorize_time(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

chargingstationdata['Time Category'] = chargingstationdata['Charging Hour'].apply(categorize_time)

# Initialize the Dash app
app = dash.Dash(__name__)

# Define the layout
app.layout = html.Div([
    html.H1("EV Charging Patterns Dashboard", style={'text-align': 'center'}),
    
    html.Div([
        html.Div([
            dcc.Dropdown(
                id='location_filter',
                options=[{'label': loc, 'value': loc} for loc in chargingstationdata['Charging Station Location'].unique()],
                value=['Houston', 'San Francisco', 'Los Angeles', 'New York', 'Chicago'],
                multi=True,
                placeholder="Select Locations"
            )
        ], style={'width': '48%', 'display': 'inline-block'}),
        
        html.Div([
            dcc.Dropdown(
                id='user_type_filter',
                options=[{'label': utype, 'value': utype} for utype in chargingstationdata['User Type'].unique()],
                value=chargingstationdata['User Type'].unique(),
                multi=True,
                placeholder="Select User Types"
            )
        ], style={'width': '48%', 'float': 'right', 'display': 'inline-block'})
    ]),
    
    html.Div([
        dcc.Graph(id='energy_consumption_by_vehicle'),
        dcc.Graph(id='charging_duration_by_time')
    ], style={'display': 'flex', 'flex-direction': 'row'}),
    
    html.Div([
        dcc.Graph(id='charging_cost_by_location'),
        dcc.Graph(id='charging_rate_by_charger_type')
    ], style={'display': 'flex', 'flex-direction': 'row'}),
    
    html.Div([
        dcc.Graph(id='charging_sessions_by_hour'),
        dcc.Graph(id='state_of_charge_changes')
    ], style={'display': 'flex', 'flex-direction': 'row'}),
    
    html.Div([
        dcc.Graph(id='distance_vs_energy'),
        dcc.Graph(id='temperature_impact')
    ], style={'display': 'flex', 'flex-direction': 'row'}),
    
    html.Div([
        dcc.Graph(id='vehicle_age_impact'),
        dcc.Graph(id='battery_capacity_distribution')
    ], style={'display': 'flex', 'flex-direction': 'row'})
])

# Callback for updating graphs based on filters
@app.callback(
    [Output('energy_consumption_by_vehicle', 'figure'),
     Output('charging_duration_by_time', 'figure'),
     Output('charging_cost_by_location', 'figure'),
     Output('charging_rate_by_charger_type', 'figure'),
     Output('charging_sessions_by_hour', 'figure'),
     Output('state_of_charge_changes', 'figure'),
     Output('distance_vs_energy', 'figure'),
     Output('temperature_impact', 'figure'),
     Output('vehicle_age_impact', 'figure'),
     Output('battery_capacity_distribution', 'figure')],
    [Input('location_filter', 'value'),
     Input('user_type_filter', 'value')]
)
def update_graphs(selected_locations, selected_user_types):
    # Filter data based on selections
    filtered_df = chargingstationdata[
        (chargingstationdata['Charging Station Location'].isin(selected_locations)) & 
        (chargingstationdata['User Type'].isin(selected_user_types))
    ]
    
    # 1. Energy Consumption by Vehicle Model
    fig1 = px.box(
        filtered_df, 
        x='Vehicle Model', 
        y='Energy Consumed (kWh)', 
        title='Energy Consumption by Vehicle Model',
        color='Vehicle Model'
    )
    fig1.update_layout(showlegend=False)
    
    # 2. Charging Duration by Time of Day
    fig2 = px.box(
        filtered_df, 
        x='Time Category', 
        y='Charging Duration (hours)', 
        title='Charging Duration by Time of Day',
        color='Time Category'
    )
    fig2.update_layout(showlegend=False)
    
    # 3. Charging Cost by Location
    fig3 = px.box(
        filtered_df, 
        x='Charging Station Location', 
        y='Charging Cost (USD)', 
        title='Charging Cost by Location',
        color='Charging Station Location'
    )
    fig3.update_layout(showlegend=False)
    
    # 4. Charging Rate by Charger Type
    fig4 = px.box(
        filtered_df, 
        x='Charger Type', 
        y='Charging Rate (kW)', 
        title='Charging Rate by Charger Type',
        color='Charger Type'
    )
    fig4.update_layout(showlegend=False)
    
    # 5. Charging Sessions by Hour of Day
    hourly_counts = filtered_df['Charging Hour'].value_counts().sort_index().reset_index()
    hourly_counts.columns = ['Hour', 'Count']
    fig5 = px.bar(
        hourly_counts, 
        x='Hour', 
        y='Count', 
        title='Charging Sessions by Hour of Day',
        color='Count',
        color_continuous_scale='Viridis'
    )
    
    # 6. State of Charge Changes
    fig6 = px.scatter(
        filtered_df,
        x='State of Charge (Start %)',
        y='State of Charge (End %)',
        title='State of Charge Changes',
        color='Vehicle Model',
        hover_data=['User Type', 'Charger Type']
    )
    
    # 7. Distance Driven vs Energy Consumed
    fig7 = px.scatter(
        filtered_df,
        x='Distance Driven (since last charge) (km)',
        y='Energy Consumed (kWh)',
        title='Distance Driven vs Energy Consumed',
        color='Vehicle Model',
        trendline="lowess"
    )
    
    # 8. Temperature Impact on Charging
    fig8 = px.scatter(
        filtered_df,
        x='Temperature (°C)',
        y='Charging Duration (hours)',
        title='Temperature Impact on Charging Duration',
        color='Vehicle Model',
        trendline="lowess"
    )
    
    # 9. Vehicle Age Impact on Charging
    fig9 = px.scatter(
        filtered_df,
        x='Vehicle Age (years)',
        y='Charging Duration (hours)',
        title='Vehicle Age Impact on Charging Duration',
        color='Vehicle Model',
        trendline="lowess"
    )
    
    # 10. Battery Capacity Distribution
    fig10 = px.histogram(
        filtered_df,
        x='Battery Capacity (kWh)',
        title='Battery Capacity Distribution',
        color='Vehicle Model',
        nbins=20
    )
    
    return fig1, fig2, fig3, fig4, fig5, fig6, fig7, fig8, fig9, fig10

# Run the app
if __name__ == '__main__':
    app.run(jupyter_mode='external', port=8052)  # Changed port to 8051

Dash app running on http://127.0.0.1:8052/
